# Phase 2 — Exploratory Data Analysis (EDA)

Goal: understand the churn dataset before building any model.
We're pulling data from the SQLite database we built in Phase 1 (not the raw CSV) — this mirrors how real teams work: the database is the single source of truth.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sqlalchemy import create_engine

# Make charts a bit nicer by default
sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
DB_PATH = PROJECT_ROOT / "data" / "processed" / "insighthub.db"

engine = create_engine(f"sqlite:///{DB_PATH}")
df = pd.read_sql("SELECT * FROM customers", engine)
print(df.shape)
df.head()

## 1. First look: shape, types, missing values

In [ ]:
df.info()

In [ ]:
# How many missing values per column?
df.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
df.describe()

## 2. The target variable: churn
This is what we're eventually trying to predict. Always look at class balance early — it changes how you evaluate a model later.

In [ ]:
churn_counts = df["churn"].value_counts()
print(churn_counts)
print((churn_counts / len(df) * 100).round(1))

sns.countplot(data=df, x="churn")
plt.title("Customer Churn Distribution")
plt.show()

**Note:** if one class is much bigger than the other (e.g. 73% vs 27%), that's class imbalance. Keep this in mind — plain accuracy will be a misleading metric later, and we'll use precision/recall/ROC-AUC instead.

## 3. Numeric feature distributions

In [ ]:
numeric_cols = ["tenure", "monthlycharges", "totalcharges"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 4. Churn rate by key categorical features
This is the visual version of the SQL queries you already ran.

In [ ]:
categorical_cols = ["contract", "internetservice", "paymentmethod", "seniorcitizen"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, categorical_cols):
    churn_rate = df.groupby(col)["churn"].apply(lambda x: (x == "Yes").mean() * 100)
    churn_rate.sort_values(ascending=False).plot(kind="bar", ax=ax)
    ax.set_title(f"Churn Rate by {col}")
    ax.set_ylabel("Churn %")

plt.tight_layout()
plt.show()

## 5. Numeric features vs. churn
Do tenure and monthly charges look different between churned and non-churned customers?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=df, x="churn", y="tenure", ax=axes[0])
axes[0].set_title("Tenure by Churn")

sns.boxplot(data=df, x="churn", y="monthlycharges", ax=axes[1])
axes[1].set_title("Monthly Charges by Churn")

plt.tight_layout()
plt.show()

## 6. Correlation heatmap (numeric features only)

In [ ]:
numeric_df = df[numeric_cols].copy()
corr = numeric_df.corr()

sns.heatmap(corr, annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Between Numeric Features")
plt.show()

Month-to-month customers churn much more frequently than customers on one-year or two-year contracts, suggesting that contract commitment is strongly associated with retention.
Newer customers with low tenure are more likely to churn than long-term customers, indicating that the early customer lifecycle is a high-risk period.
Customers with higher monthly charges tend to show higher churn, suggesting that pricing or perceived value may be important factors in retention.
Internet service, payment method, and additional services such as Tech Support and Online Security show noticeable differences in churn rates, making them useful variables for customer segmentation.
Churn is moderately imbalanced—roughly 27% of customers churn—so accuracy alone would not be a good evaluation metric; precision, recall, F1, ROC-AUC, PR-AUC, and business impact should also be considered.